In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# CUSTOM CNN FROM SCRATCH — CENTER ROI REGRESSION
# Subsets: white paper / tables / white+tables
# Lux band: 150–700
# ============================================================
from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# CONFIG
# ============================================================
FEATURE_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/fixed_parser_refresh/extracted_features_refresh_5roi_fullcontext_fixed.csv")

OUTDIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/custom_cnn_center_regression")
OUTDIR.mkdir(parents=True, exist_ok=True)

SUBSETS_TO_RUN = ["white_paper_only", "tables_only", "white_plus_tables"]

LUX_MIN = 150
LUX_MAX = 700

TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.15
RANDOM_STATE = 42

ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}
ROI_NAME = "C"
ROI_DIAMETER = 256

IMAGE_SIZE = 128       # smaller than ResNet, good for a custom CNN
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_LOG_TARGET = True
LOSS_NAME = "huber"    # "mse" or "huber"
SEED = 42

# ============================================================
# REPRODUCIBILITY
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# HELPERS
# ============================================================
def apply_subset(df: pd.DataFrame, subset_name: str) -> pd.DataFrame:
    if subset_name == "all_rows":
        return df.copy()
    elif subset_name == "white_paper_only":
        return df[df["surface_group"] == "white_paper"].copy()
    elif subset_name == "tables_only":
        return df[df["surface_group"] == "table"].copy()
    elif subset_name == "white_plus_tables":
        return df[df["surface_group"].isin(["white_paper", "table"])].copy()
    elif subset_name == "colored_paper_only":
        return df[df["surface_group"] == "colored_paper"].copy()
    else:
        raise ValueError(f"Unknown subset: {subset_name}")

def open_rgb_exif_safe(image_path: str) -> Image.Image:
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    return img

def crop_patch(img: Image.Image, center_xy, diameter_px: int) -> Image.Image:
    x, y = center_xy
    r = diameter_px // 2
    arr = np.array(img)
    h, w = arr.shape[:2]
    x0, x1 = max(0, x - r), min(w, x + r)
    y0, y1 = max(0, y - r), min(h, y + r)
    patch = arr[y0:y1, x0:x1].copy()
    return Image.fromarray(patch)

def transform_target(y):
    return np.log(y) if USE_LOG_TARGET else y

def inverse_target(yhat):
    return np.exp(yhat) if USE_LOG_TARGET else yhat

def metrics_dict(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    eps = 1e-9
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100.0),
    }

# ============================================================
# DATASET
# ============================================================
class LuxROIDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None, roi_name="C", roi_diameter=256):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.roi_name = roi_name
        self.roi_diameter = roi_diameter

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = open_rgb_exif_safe(row["image_path"])
        patch = crop_patch(img, ROI_COORDS[self.roi_name], self.roi_diameter)

        if self.transform is not None:
            patch = self.transform(patch)

        y = float(row["target_lux"])
        y = transform_target(y)
        y = torch.tensor([y], dtype=torch.float32)
        return patch, y

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.08),
    transforms.ToTensor(),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

# ============================================================
# SMALL CNN FROM SCRATCH
# ============================================================
class SmallLuxCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 128 -> 64

            # block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32

            # block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16

            # block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

def make_model():
    return SmallLuxCNN().to(DEVICE)

# ============================================================
# TRAIN / EVAL
# ============================================================
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)

    total_loss = 0.0
    y_true_all = []
    y_pred_all = []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        pred = model(xb)
        loss = criterion(pred, yb)

        if train_mode:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * xb.size(0)
        y_true_all.extend(yb.detach().cpu().numpy().reshape(-1).tolist())
        y_pred_all.extend(pred.detach().cpu().numpy().reshape(-1).tolist())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(y_true_all), np.array(y_pred_all)

# ============================================================
# LOAD DATA
# ============================================================
df0 = pd.read_csv(FEATURE_CSV)

required_cols = {"image_path", "image_name", "session", "surface_group", "target_lux"}
missing = required_cols - set(df0.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

all_summary_rows = []

for subset_name in SUBSETS_TO_RUN:
    print("\n" + "=" * 90)
    print("Subset:", subset_name)

    df = apply_subset(df0, subset_name).reset_index(drop=True)
    df = df[(df["target_lux"] >= LUX_MIN) & (df["target_lux"] <= LUX_MAX)].copy().reset_index(drop=True)

    print("Rows after lux filter:", len(df))
    if len(df) < 50:
        print("Skipping: too few rows")
        continue

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(df, groups=df["session"].astype(str).values))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE_WITHIN_TRAIN, random_state=RANDOM_STATE)
    fit_idx, val_idx = next(gss2.split(train_df, groups=train_df["session"].astype(str).values))
    fit_df = train_df.iloc[fit_idx].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)

    print("Fit rows:", len(fit_df), "| Val rows:", len(val_df), "| Test rows:", len(test_df))

    train_ds = LuxROIDataset(fit_df, transform=train_tf, roi_name=ROI_NAME, roi_diameter=ROI_DIAMETER)
    val_ds   = LuxROIDataset(val_df, transform=eval_tf, roi_name=ROI_NAME, roi_diameter=ROI_DIAMETER)
    test_ds  = LuxROIDataset(test_df, transform=eval_tf, roi_name=ROI_NAME, roi_diameter=ROI_DIAMETER)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = make_model()

    if LOSS_NAME == "mse":
        criterion = nn.MSELoss()
    else:
        criterion = nn.HuberLoss(delta=0.5)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    best_val_mae = np.inf
    best_state = None
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, y_true_tr_t, y_pred_tr_t = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, y_true_va_t, y_pred_va_t = run_epoch(model, val_loader, criterion, optimizer=None)

        y_true_tr = inverse_target(y_true_tr_t)
        y_pred_tr = inverse_target(y_pred_tr_t)
        y_true_va = inverse_target(y_true_va_t)
        y_pred_va = inverse_target(y_pred_va_t)

        train_mae = mean_absolute_error(y_true_tr, y_pred_tr)
        val_mae = mean_absolute_error(y_true_va, y_pred_va)

        history.append({
            "subset": subset_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_MAE": train_mae,
            "val_MAE": val_mae,
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
            f"train_MAE={train_mae:.2f} val_MAE={val_mae:.2f}"
        )

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)

    test_loss, y_true_te_t, y_pred_te_t = run_epoch(model, test_loader, criterion, optimizer=None)
    y_true_te = inverse_target(y_true_te_t)
    y_pred_te = inverse_target(y_pred_te_t)

    metrics = metrics_dict(y_true_te, y_pred_te)
    print("TEST:", metrics)

    hist_df = pd.DataFrame(history)
    pred_df = test_df[["image_name", "session", "surface_group", "target_lux"]].copy()
    pred_df["y_true"] = y_true_te
    pred_df["y_pred"] = y_pred_te
    pred_df["abs_err"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])
    pred_df["pct_err"] = np.abs((pred_df["y_true"] - pred_df["y_pred"]) / np.maximum(np.abs(pred_df["y_true"]), 1e-9)) * 100.0

    hist_path = OUTDIR / f"history__{subset_name}__{LUX_MIN}_{LUX_MAX}.csv"
    pred_path = OUTDIR / f"predictions__{subset_name}__{LUX_MIN}_{LUX_MAX}.csv"
    model_path = OUTDIR / f"smallcnn_regressor__{subset_name}__{LUX_MIN}_{LUX_MAX}.pt"

    hist_df.to_csv(hist_path, index=False)
    pred_df.to_csv(pred_path, index=False)
    torch.save({
        "model_state_dict": model.state_dict(),
        "subset": subset_name,
        "lux_min": LUX_MIN,
        "lux_max": LUX_MAX,
        "roi_name": ROI_NAME,
        "roi_diameter": ROI_DIAMETER,
        "image_size": IMAGE_SIZE,
        "use_log_target": USE_LOG_TARGET,
    }, model_path)

    all_summary_rows.append({
        "subset": subset_name,
        "model": "SmallCNN_from_scratch",
        "lux_min": LUX_MIN,
        "lux_max": LUX_MAX,
        "fit_n": len(fit_df),
        "val_n": len(val_df),
        "test_n": len(test_df),
        **metrics,
    })

summary_df = pd.DataFrame(all_summary_rows)
summary_path = OUTDIR / f"summary__{LUX_MIN}_{LUX_MAX}.csv"
summary_df.to_csv(summary_path, index=False)

print("\nSaved summary:")
print(summary_path)
display(summary_df)


Subset: white_paper_only
Rows after lux filter: 371
Fit rows: 251 | Val rows: 45 | Test rows: 75
Epoch 01 | train_loss=2.4612 val_loss=2.4968 | train_MAE=410.10 val_MAE=387.89
Epoch 02 | train_loss=1.0754 val_loss=0.7247 | train_MAE=376.11 val_MAE=280.03
Epoch 03 | train_loss=0.4941 val_loss=0.9716 | train_MAE=6624.46 val_MAE=335.13
Epoch 04 | train_loss=0.3212 val_loss=0.2712 | train_MAE=334.05 val_MAE=266.21
Epoch 05 | train_loss=0.2230 val_loss=0.1968 | train_MAE=378.98 val_MAE=644.63
Epoch 06 | train_loss=0.2447 val_loss=0.1095 | train_MAE=360.24 val_MAE=139.60
Epoch 07 | train_loss=0.2246 val_loss=0.0866 | train_MAE=494.98 val_MAE=161.63
Epoch 08 | train_loss=0.2048 val_loss=0.0481 | train_MAE=253.59 val_MAE=95.48
Epoch 09 | train_loss=0.1963 val_loss=0.0196 | train_MAE=351.11 val_MAE=65.69
Epoch 10 | train_loss=0.1990 val_loss=0.0616 | train_MAE=260.33 val_MAE=101.35
Epoch 11 | train_loss=0.2060 val_loss=0.0833 | train_MAE=307.19 val_MAE=124.24
Epoch 12 | train_loss=0.2079 val_l

,subset,model,lux_min,lux_max,fit_n,val_n,test_n,MAE,RMSE,R2,MAPE
0,white_paper_only,SmallCNN_from_scratch,150,700,251,45,75,100.491600,155.020518,0.129115,21.585098
1,tables_only,SmallCNN_from_scratch,150,700,163,29,49,94.929501,130.437127,0.259242,23.089733
2,white_plus_tables,SmallCNN_from_scratch,150,700,415,72,125,73.506572,106.079488,0.591232,17.804117
